In [ ]:
import re
import gc
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print
print(" HIGH-PERFORMANCE TWO-PHASE SOURCE TREE ENGINE")


# 1. LOAD MODEL (HIGHLY OPTIMIZED 4-BIT PRECISION)

MODEL_NAME = "ibm-granite/granite-3.3-8b-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="sdpa", 
    trust_remote_code=True
)
model.eval() 
print(" Granite-3.3-8B successfully armed in quantized layout precision.")

In [ ]:





#  PHYSICAL DATASET AND CODE DIRECTORY PATH PATTERNS

RSF_PATH = "    "       #  path of the .rsf file 

#  FIXED PATH: We point this to the base 'java' directory so package parts match up perfectly
SRC_ROOT_DIR = "      "   #  path of the java file

# Parse RSF structural dataset boundaries
clusters = {}
with open(RSF_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 3 and parts[0] == "contain":
            cluster, classname = parts[1], parts[2]
            clusters.setdefault(cluster, []).append(classname)

for k in clusters:
    clusters[k] = sorted(list(set(clusters[k])))

print(f" Structural dataset loaded successfully. Discovered {len(clusters)} clusters in RSF target.")

def query_llm(prompt, tokens_limit=400):
    messages = [
        {"role": "system", "content": "You are a concise software architect. Always finish sentences completely."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=3500).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=tokens_limit, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Helper to find where a java file is physically located under your source root package
def map_class_to_file(classname):
    clean_name = classname.split("$")[0] # Clear inner classes
    path_parts = clean_name.split(".")
    relative_path = Path(*path_parts[:-1]) / f"{path_parts[-1]}.java"
    
    possible_path = Path(SRC_ROOT_DIR) / relative_path
    if possible_path.exists():
        return possible_path
        
    # Fallback debug search: Check if it's directly inside the apache/hadoop subfolder tracks
    fallback_path = Path("   ") / Path(*path_parts[4:-1]) / f"{path_parts[-1]}.java"
    if fallback_path.exists():
        return fallback_path
        
    return None

# ====================================================
# 2. FILE SUMMARIZATION PROCESS ENGINE
# ====================================================
def summarize_cluster(cluster_id, classes_list):
    print(f"\n High-Speed Package-Tree Roll-Up for {cluster_id}")
    
    # Map out files matching active directory groupings
    package_groups = {}
    found_count = 0
    for c in classes_list:
        file_path = map_class_to_file(c)
        if file_path:
            package_groups.setdefault(file_path.parent, []).append(file_path)
            found_count += 1

    print(f"       Found {found_count} out of {len(classes_list)} physical .java files on disk.")

    if not package_groups:
        print(f"       ALERT: No physical source files found for {cluster_id}. Generating architectural fallback baseline summary.")
        fallback_prompt = f"Write a dense software architecture summary paragraph under 110 words for a Hadoop MapReduce system cluster partition containing these components: {', '.join(classes_list[:15])}. Focus on distributed computing roles, technologies used, scalability, and performance."
        fallback_summary = query_llm(fallback_prompt, tokens_limit=250)
        return f"Title: MapReduce Distributed Subsystem\nDescription: {fallback_summary}"

    # Sort target folders by depth in reverse order (deepest folders processed first!)
    sorted_packages = sorted(package_groups.keys(), key=lambda x: len(x.parts), reverse=True)
    
    # Clear context loops for EVERY independent cluster run to eliminate variable leakage
    running_branch_context = "   "

    # Loop through directories from bottom up
    for package_folder in sorted_packages:
        local_files = package_groups[package_folder]
        local_summaries = []
        
        # PHASE 1: LEAF NODE SUMMARIZATION (Read Raw Code)
        
        # Cap at top 5 files per directory layer to maximize processing speed
        for java_file in local_files[:5]:
            try:
                print(f"          Processing: {java_file.name}")
                with open(java_file, "r", encoding="utf-8", errors="ignore") as f:
                    raw_code = f.read()[:4000] # Safe 4,000 char processing cap window
                
                leaf_prompt = f"Summarize this Java source file {java_file.name} under 25 words focusing purely on core behavior:\n{raw_code}"
                file_summary = query_llm(leaf_prompt, tokens_limit=45)
                local_summaries.append(f"File {java_file.name}: {file_summary}")
            except Exception as e:
                print(f"          Error reading {java_file.name}: {str(e)}")
                continue

        #  PHASE 2: BRANCH NODE SUMMARIZATION (Aggregate Summaries)
        if local_summaries:
            combined_local_text = "\n".join(local_summaries)
            branch_prompt = f"Summarize the collective duties of directory package '{package_folder.name}' in 35 words based on these file notes:\n{combined_local_text}"
            
            if running_branch_context:
                branch_prompt += f"\n\nIntegrate deeper sub-directory context history layers here:\n{running_branch_context}"
            
            running_branch_context = query_llm(branch_prompt, tokens_limit=65)

    # FINAL ROOT MODULE SYNTHESIS
    root_prompt = f"""You are an expert software engineer performing architectural recovery on Apache Hadoop MapReduce.
    The complete bottom-up physical package structure summary for the cluster {cluster_id} has been compiled below:
    
    Package Tree Context:
    {running_branch_context}
    
    Generate exactly the following output layout:
    Title: [A short 3-to-5 word functional module title]
    Description: [A single continuous summary paragraph that explicitly mentions major components, interactions, technologies used (Java, Hadoop, HDFS), scalability, performance, and fault tolerance. Keep this under 130 words.]
    """
    
    output = query_llm(root_prompt, tokens_limit=350)
    
    # Word-Count Self-Correction Post-Processor Engine
    desc_match = re.search(r"Description:\s*(.*)", output, re.DOTALL | re.IGNORECASE)
    description = desc_match.group(1).strip() if desc_match else output
    word_count = len(description.split())
    
    if word_count > 150:
        compression_prompt = f"Rewrite this description into a continuous paragraph under 130 words total. Retain all quality attributes:\n{description}"
        compressed_desc = query_llm(compression_prompt, tokens_limit=200)
        compressed_desc = re.sub(r"^(Description:)\s*", "", compressed_desc, flags=re.IGNORECASE)
        output = f"Title: {output.split('Description:')[0].replace('Title:', '').strip()}\nDescription: {compressed_desc.strip()}"

    return output

# 3. RUN PIPELINE ENGINE OVER ALL CLUSTERS
results = []
total_clusters = len(clusters)

for idx, (cluster_id, classes) in enumerate(clusters.items(), start=1):
    print(f"[{idx}/{total_clusters}] Accelerating Module: {cluster_id}")
    output = summarize_cluster(cluster_id, classes)
    
    title_match = re.search(r"Title:\s*(.*?)\n", output, re.IGNORECASE)
    desc_match = re.search(r"Description:\s*(.*)", output, re.DOTALL | re.IGNORECASE)
    
    title = title_match.group(1).strip() if title_match else f"MapReduce Module - {cluster_id}"
    description = desc_match.group(1).strip() if desc_match else output

    title = title.replace('**', '').replace('`', '').strip()
    description = description.replace('**', '').replace('`', '').strip()

    results.append({"cluster_ID": cluster_id, "files": ", ".join(classes), "title": title, "description": description})
    
    # Save spreadsheet entries dynamically step-by-step
    pd.DataFrame(results).to_csv("ARC_OUTPUT_FINAL.csv", index=False)
    
    gc.collect()
    torch.cuda.empty_cache()

print("compliant spreadsheet: LIMBO_OUTPUT_FINAL.csv")